Import Libraries.

In [100]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from wildfireAnalyticsBackEnd import readDirectory as rd

The folder `HistorialWildfirePerimeters` contains data on all of the wildfires in Alberta that occured from 1931 to 2022.

The folder `NaturalEarthData` contains data on all of the states and provinces in the world. For our purposes, we are only interested in the state of Alberta, Canada.

Obtain the shapefiles these folders and read them into `wildfires` and `statesProvinces` respectively.

In [ ]:
# For the wildfires
wildfiresDir = "HistoricalWildfirePerimeters"
wildfiresShp = rd.get_shape_file(wildfiresDir)
wildfires = gpd.read_file(wildfiresShp)

# For all of the states and provinces in the world
statesProvincesDir = "NaturalEarthData"
statesProvincesShp = rd.get_shape_file(statesProvincesDir)
statesProvinces = gpd.read_file(statesProvincesShp)

Since the wildfire data is only for those located in Alberta, we want to Obtain the data for just Alberta, Canada from `statesProvince`.

In [ ]:
alberta = statesProvinces[statesProvinces["iso_3166_2"] == "CA-AB"]

Plot the `wildfires` map and the `alberta` map together.

In [ ]:
basePlot = alberta.plot(alpha=0.5)
wildfires.plot(ax=basePlot.axes, alpha=0.5)

The scale looks terrible. This is because their Coordinate Reference Systems (CRS) do not match. To make the scale easier to view, we'll set their CRS to EPSG 4326, which is a commonly-used CRS system. After we do that, we'll plot the maps together again.

In [ ]:
# Save the Crs first if we need to revert back
wildfiresCrs = wildfires.crs
albertaCrs = alberta.crs

# the line below is unnecessary since alberta.crs = "EPSG:4326" already.
# alberta = alberta.to_crs("EPGS:4326")
wildfires = wildfires.to_crs(alberta.crs)

basePlot = alberta.plot(alpha=0.5)
wildfires.plot(ax=basePlot.axes, alpha=0.5)

The image is much clearer since both are using the same CRS. However, let's write a script to see if we can get the wildfires to fall perfectly within with the state lines.

In [ ]:
# the portion of wildfires that overlaps with alberta
wildfiresInside = gpd.overlay(alberta, wildfires, how="intersection")

# plot this overlap with the alberta map
basePlot = alberta.plot(alpha=0.5)
wildfiresInside.plot(ax=basePlot.axes, alpha=0.5)

This is great! However, this shows every fire from the years 1931 to 2022. What if I want to show a specific year, such as 2002?

In [ ]:
wildfireYear2002 = wildfiresInside[wildfiresInside["YEAR"] == 2002]

basePlot = alberta.plot(alpha=0.5)
wildfireYear2002.plot(ax=basePlot.axes, alpha=0.5)

The darker spots are wildfires that occured in Alberta. Let's show ALL wildfires in the 21st century.

In [ ]:
wildfiresYear2000Onwards = wildfiresInside[wildfiresInside["YEAR"] >= 2000]

basePlot = alberta.plot(alpha=0.5)
wildfiresYear2000Onwards.plot(ax=basePlot.axes, alpha=0.5)

Great! I want to analyze the wildfire in row 4384. More specifically, I want to see how much land was burnt from the wildfire.

In [ ]:
# set wildfires2000Onwards to its previous CRS.
wildfiresYear2000Onwards = wildfiresYear2000Onwards.to_crs(wildfiresCrs)
wildfiresYear2000Onwards.head()

# Area from the data itself
area_squnits = wildfiresYear2000Onwards.loc[4384, "SHAPE_STAr"]
area_hectares = wildfiresYear2000Onwards.loc[4384, "HECTARES_U"]
print("Area in square units (m^2):", area_squnits)
print("Area in hectares:", area_hectares)

# Area from gpd.area
area_squnits = wildfiresYear2000Onwards.loc[4384, "geometry"].area
area_hectares = area_squnits / 10000  # Since 1 hectare = 10,000 square meters
print("Area in square units (m^2):", area_squnits)
print("Area in hectares:", area_hectares)

What can we do with this? Well, let's do something simple: figuring out how much land was burned each year from 2000 onwards.

In [ ]:
# Store the data using hectares
landBurned2000Onwards = np.zeros(23)

for year in range(2000, 2022):
    # get all the wildfires in a specific year
    wildfiresYear = wildfiresYear2000Onwards[wildfiresYear2000Onwards["YEAR"] == year]

    # for each row in wildfiresYear
    for index, row in wildfiresYear.iterrows():
        area_hectares = row["HECTARES_U"]
        landBurned2000Onwards[year-2000] += area_hectares


We have stored the data in a numpy array, where `landBurned2000Onwards[0]` is the total area in year 2000, `landBurned2000Onwards[1]` is the total area in year 2001, and so on.

In [ ]:
# x-axis
years = np.arange(2000, 2023)

plt.plot(years, landBurned2000Onwards/1000, marker='o', linestyle='-')

# Add labels and a title
plt.xlabel("Year")
plt.ylabel("Land Burned: kiloHectares")
plt.title("Total Land Burned in Alberta from 2000 to 2022")

# Show the plot
plt.grid()  # Add grid lines (optional)
plt.show()

Now, I want to use DBSCAN to cluster the polygons together

In [102]:
# Extract the polygons from the geometries
wildfireCentroids = wildfiresYear2000Onwards.geometry.centroid
print(wildfireCentroids)
print(type(wildfireCentroids))
coordinates = [(point.x, point.y) for point in wildfireCentroids]

eps = 0.1
min_samples = 5
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
cluster_labels = dbscan.fit_predict(coordinates)

# Add cluster labels to the original GeoDataFrame
wildfiresYear2000Onwards['cluster'] = cluster_labels

# Visualize the clusters
wildfiresYear2000Onwards.plot(column='cluster', legend=True)

4384     POINT (715267.923 6312450.891)
4385     POINT (734114.616 6260399.721)
4386     POINT (705746.847 6644204.720)
4387     POINT (708727.521 6646690.363)
4388     POINT (709669.861 6644228.985)
                      ...              
37332    POINT (384913.321 6005466.812)
37333    POINT (343677.435 6039468.012)
37334    POINT (415989.441 6032952.629)
37335    POINT (466168.215 5982011.222)
37336    POINT (419017.165 6010426.083)
Length: 32953, dtype: geometry
<class 'geopandas.geoseries.GeoSeries'>


Additional Lines of Code.

In [ ]:
'''
# get area of polygon
area_squnits = data.loc[4495, "geometry"].area

# get perimeter length of polygon
perim = data.loc[4495, "geometry"].length

area_hectares = area_squnits / 10000  # Since 1 hectare = 10,000 square meters

print("Area in square units:", area_squnits)
print("Area in hectares:", area_hectares)  # same as hectares area
print("Perimeter length:", perim)
print("points of 4495:", data.loc[4995, "geometry"])
print("type of previous object:", type(data.loc[4995, "geometry"]))

# How to print all the data
data.head()

# print row
row = data.iloc[4495]
print(row)

# Get CRS (Coordinate-Reference System)
crsInfo = data.crs
print("CRS is ", crsInfo)

# Print only the column titles
headers = data.columns
print(headers)

# Access Data only from Year 2000 and onwards
dataYear2000Onwards = data[data["YEAR"] >= 2000]
print(dataYear2000Onwards)


print(type(canadaStates))
print(canadaStates)

alberta = statesProvinces.iloc[1239]
print(alberta)
print(type(alberta))
'''